# Experiment: CMNIST DGIL Full Pipeline

Objective:
- Reproduce the CMNIST DGIL-style workflow end-to-end in one notebook.
- Run a quick paper-aligned IRO pipeline by default, with optional full comparison sweep.
- Analyze locally trained checkpoints and externally copied cluster checkpoints (`.pkl`) without SLURM.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
import math
import os
import shlex
import time
from collections import defaultdict
from pathlib import Path
from statistics import mean, stdev

import numpy as np
import torch

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

from iro import run_evaluation, run_training

GLOBAL_SEED = 0
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
GLOBAL_SEED


## Configuration and Toggles

Use these toggles to control runtime and output layout.


In [ ]:
# Core toggles
QUICK_MODE = True
RUN_FULL_SWEEP = False

# Paths
OUTPUT_ROOT = "notebooks/artifacts"
DATA_ROOT = "data/cmnist"
DOWNLOAD_DATA = False

# Optional analysis from externally trained artifacts
USE_EXTERNAL_PICKLES = True
EXTERNAL_CKPT_DIR = "notebooks/imported_checkpoints"   # copy *_final.pkl and *_best.pkl here
EXTERNAL_RESULTS_DIR = "notebooks/imported_results"     # optional: copy JSONL result files here

# Sweep controls
N_SEEDS = 10  # set to 10 to mimic ARRAY_RANGE=0-9
FULL_SWEEP_LIMIT = None  # optional debug cap, e.g. 8

# Evaluation and model selection
MODEL_SELECTION_ENV = 0.9
MODEL_SELECTION_TYPE = "best"  # "best" or "final"
TEST_ENVS_ALL = [i / 10.0 for i in range(11)]

# Quick path defaults
QUICK_EXP_NAME = "cmnist_quick_paper_iro"
QUICK_STEPS_OVERRIDE = None  # e.g. 60 for a faster smoke run
EVAL_ALPHA = 0.8

# Full sweep defaults
FULL_EXP_PREFIX = "cmnist_full_repro"


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not find repository root from current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_ROOT_PATH = (REPO_ROOT / OUTPUT_ROOT).resolve()
DATA_ROOT_PATH = (REPO_ROOT / DATA_ROOT).resolve()
EXTERNAL_CKPT_DIR_PATH = (REPO_ROOT / EXTERNAL_CKPT_DIR).resolve()
EXTERNAL_RESULTS_DIR_PATH = (REPO_ROOT / EXTERNAL_RESULTS_DIR).resolve()

PAPER_OVERRIDES_FILE = REPO_ROOT / "scripts/overrides/cmnist_paper_repro.txt"
FULL_GRID_BASE_FILE = REPO_ROOT / "scripts/overrides/cmnist_full_grid_base.txt"

OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)
EXTERNAL_CKPT_DIR_PATH.mkdir(parents=True, exist_ok=True)
EXTERNAL_RESULTS_DIR_PATH.mkdir(parents=True, exist_ok=True)

print("repo_root:", REPO_ROOT)
print("output_root:", OUTPUT_ROOT_PATH)
print("data_root:", DATA_ROOT_PATH)
print("external_ckpt_dir:", EXTERNAL_CKPT_DIR_PATH)
print("external_results_dir:", EXTERNAL_RESULTS_DIR_PATH)


## Parse Paper and Base Override Files

This keeps notebook config aligned with repository-provided experiment specs.


In [ ]:
def parse_override_file(path: Path) -> list[str]:
    if not path.exists():
        raise FileNotFoundError(f"Override file not found: {path}")

    overrides: list[str] = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        overrides.append(line)
    return overrides


paper_overrides = parse_override_file(PAPER_OVERRIDES_FILE)
full_grid_base_overrides = parse_override_file(FULL_GRID_BASE_FILE)

print("paper_overrides:")
for item in paper_overrides:
    print("  ", item)

print("\nfull_grid_base_overrides:")
for item in full_grid_base_overrides:
    print("  ", item)


## Full Sweep Definition (84 Configurations Before Seeds)

This reproduces the same grid construction as `scripts/submit_cmnist_full_grid.sh`.


In [ ]:
def build_full_grid_configs(exp_prefix: str) -> list[dict]:
    penalties = [1000, 5000, 10000, 50000, 100000]
    sd_penalties = [1, 5, 10, 50, 100]
    groupdro_etas = [0.001, 0.01, 0.1, 0.5, 1.0]
    eqrm_alphas = [-100, -500, -1000, -5000, -10000]
    iid_alphas = [0.2, 0.4, 0.6, 0.8, 0.95]
    erm_pretrain_steps = [0, 400]
    erm_total_steps = [400, 600, 1000]

    configs: list[dict] = []

    # ERM + gray-ERM oracle row
    for train_env in ["default", "gray"]:
        for steps in erm_total_steps:
            exp_name = f"{exp_prefix}_erm_{train_env}_s{steps}"
            extra = [
                "iro.algorithm=erm",
                f"data.cmnist_train_envs={train_env}",
                f"training.steps={steps}",
                "training.erm_pretrain_iters=0",
                "training.lr_cos_sched=false",
                "training.save_ckpts=false",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

    # Non-ERM algorithms
    for pretrain in erm_pretrain_steps:
        for eta in groupdro_etas:
            exp_name = f"{exp_prefix}_groupdro_pre{pretrain}_eta{eta}"
            extra = [
                "iro.algorithm=groupdro",
                "training.steps=1000",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.groupdro_eta={eta}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for pen in sd_penalties:
            exp_name = f"{exp_prefix}_sd_pre{pretrain}_pen{pen}"
            extra = [
                "iro.algorithm=sd",
                "training.steps=1000",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.penalty_weight={pen}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for pen in penalties:
            exp_name = f"{exp_prefix}_iga_pre{pretrain}_pen{pen}"
            extra = [
                "iro.algorithm=iga",
                "training.steps=1000",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.penalty_weight={pen}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for pen in penalties:
            exp_name = f"{exp_prefix}_irm_pre{pretrain}_pen{pen}"
            extra = [
                "iro.algorithm=irm",
                "training.steps=600",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.penalty_weight={pen}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for pen in penalties:
            exp_name = f"{exp_prefix}_vrex_pre{pretrain}_pen{pen}"
            extra = [
                "iro.algorithm=vrex",
                "training.steps=600",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.penalty_weight={pen}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for alpha in eqrm_alphas:
            exp_name = f"{exp_prefix}_eqrm_pre{pretrain}_a{alpha}"
            extra = [
                "iro.algorithm=eqrm",
                "training.steps=600",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.alpha={alpha}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for steps in [600, 1000]:
            exp_name = f"{exp_prefix}_iro_pre{pretrain}_s{steps}"
            extra = [
                "iro.algorithm=iro",
                f"training.steps={steps}",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for steps in [600, 1000]:
            exp_name = f"{exp_prefix}_inftask_pre{pretrain}_s{steps}"
            extra = [
                "iro.algorithm=inftask",
                f"training.steps={steps}",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

        for alpha in iid_alphas:
            exp_name = f"{exp_prefix}_iid_pre{pretrain}_a{alpha}"
            extra = [
                "iro.algorithm=iid",
                "training.steps=600",
                f"training.erm_pretrain_iters={pretrain}",
                "training.lr_cos_sched=true",
                "training.save_ckpts=true",
                f"iro.alpha={alpha}",
            ]
            configs.append({"exp_name": exp_name, "extra_overrides": extra})

    if len(configs) != 84:
        raise RuntimeError(f"Expected 84 configs, got {len(configs)}")

    required_algs = {"erm", "eqrm", "irm", "groupdro", "vrex", "iga", "sd", "iro", "inftask", "iid"}
    seen_algs = {next(x.split("=", 1)[1] for x in cfg["extra_overrides"] if x.startswith("iro.algorithm=")) for cfg in configs}
    if required_algs - seen_algs:
        missing = sorted(required_algs - seen_algs)
        raise RuntimeError(f"Missing algorithms in grid: {missing}")

    oracle_present = any(
        any(x == "iro.algorithm=erm" for x in cfg["extra_overrides"]) and
        any(x == "data.cmnist_train_envs=gray" for x in cfg["extra_overrides"])
        for cfg in configs
    )
    if not oracle_present:
        raise RuntimeError("Expected gray ERM oracle row in sweep definition")

    return configs


def expand_with_seeds(configs: list[dict], n_seeds: int) -> list[dict]:
    jobs: list[dict] = []
    for cfg in configs:
        for seed in range(n_seeds):
            jobs.append(
                {
                    "exp_name": cfg["exp_name"],
                    "seed": seed,
                    "extra_overrides": list(cfg["extra_overrides"]),
                }
            )
    return jobs


full_grid_configs = build_full_grid_configs(FULL_EXP_PREFIX)
print("full_grid_configs:", len(full_grid_configs))
print("seed-expanded jobs:", len(expand_with_seeds(full_grid_configs, N_SEEDS)))
print("example config:", full_grid_configs[0])


## Execution Helpers

These wrappers run train/eval through the Python API and capture structured metadata.


In [ ]:
TRAIN_RUN_SUMMARIES: list[dict] = []
EVAL_RUN_SUMMARIES: list[dict] = []


def bool_str(flag: bool) -> str:
    return "true" if flag else "false"


def common_runtime_overrides(*, exp_name: str, seed: int) -> list[str]:
    return [
        f"data.root={DATA_ROOT_PATH}",
        f"data.download={bool_str(DOWNLOAD_DATA)}",
        f"training.output_root={OUTPUT_ROOT_PATH}",
        f"training.exp_name={exp_name}",
        f"training.seed={seed}",
        "training.write_artifacts=true",
        "training.capture_logs=true",
    ]


def summarize_training_result(result: dict, *, exp_name: str, seed: int) -> dict:
    artifacts = result.get("artifacts", {}) or {}
    summary = {
        "exp_name": exp_name,
        "seed": seed,
        "algorithm": result.get("algorithm_name"),
        "steps": result.get("steps"),
        "selection_env": result.get("selection_env"),
        "results_file": artifacts.get("results_file"),
        "ckpt_final": artifacts.get("ckpt_final"),
        "ckpt_best": artifacts.get("ckpt_best"),
        "run_id": artifacts.get("run_id"),
        "status": "ok",
    }
    for key, value in result.items():
        if isinstance(key, str) and (
            key.endswith("_acc_final")
            or key.endswith("_loss_final")
            or key.endswith("_acc_best")
            or key.endswith("_loss_best")
        ):
            summary[key] = float(value)
    return summary


def run_train_job(
    *,
    exp_name: str,
    seed: int,
    base_overrides: list[str],
    extra_overrides: list[str] | None = None,
    verbose: bool = True,
) -> tuple[dict, dict]:
    overrides = []
    overrides.extend(common_runtime_overrides(exp_name=exp_name, seed=seed))
    overrides.extend(base_overrides)
    if extra_overrides:
        overrides.extend(extra_overrides)

    started = time.time()
    result = run_training(experiment="cmnist_iro", overrides=overrides)
    elapsed = time.time() - started

    summary = summarize_training_result(result, exp_name=exp_name, seed=seed)
    summary["elapsed_seconds"] = elapsed
    TRAIN_RUN_SUMMARIES.append(summary)

    if verbose:
        print(
            f"[train] exp={exp_name} seed={seed} alg={summary.get('algorithm')} "
            f"steps={summary.get('steps')} elapsed={elapsed:.1f}s"
        )
        print("        results_file:", summary.get("results_file"))
        print("        ckpt_final:", summary.get("ckpt_final"))
        print("        ckpt_best:", summary.get("ckpt_best"))

    return summary, result


def run_eval_job(
    *,
    checkpoint_path: str | Path,
    algorithm: str,
    split: str,
    alpha: float,
    exp_name: str,
    seed: int = 0,
) -> dict:
    checkpoint_path = str(checkpoint_path)
    overrides = [
        f"data.root={DATA_ROOT_PATH}",
        "data.download=false",
        f"training.output_root={OUTPUT_ROOT_PATH}",
        f"training.exp_name={exp_name}",
        f"training.seed={seed}",
        f"iro.algorithm={algorithm}",
        f"eval.checkpoint_path={checkpoint_path}",
        f"eval.split={split}",
        f"eval.alpha={alpha}",
    ]
    result = run_evaluation(experiment="cmnist_iro", overrides=overrides)

    summary = {
        "exp_name": exp_name,
        "algorithm": algorithm,
        "split": split,
        "alpha": alpha,
        "checkpoint_path": checkpoint_path,
        "metrics": result.get("metrics", []),
        "results_file": (result.get("artifacts", {}) or {}).get("results_file"),
    }
    EVAL_RUN_SUMMARIES.append(summary)
    return summary


## Quick Path (Default): Paper-Aligned IRO Run + Eval

This runs one CMNIST IRO training job using the paper override file, then evaluates saved checkpoints.


In [ ]:
QUICK_RUN_ARTIFACTS = {}

if QUICK_MODE:
    quick_overrides = list(paper_overrides)
    if QUICK_STEPS_OVERRIDE is not None:
        quick_overrides.append(f"training.steps={int(QUICK_STEPS_OVERRIDE)}")

    quick_summary, quick_result = run_train_job(
        exp_name=QUICK_EXP_NAME,
        seed=0,
        base_overrides=quick_overrides,
        extra_overrides=None,
        verbose=True,
    )

    ckpt_final = quick_summary.get("ckpt_final")
    ckpt_best = quick_summary.get("ckpt_best")
    algorithm = str(quick_summary.get("algorithm") or "iro")

    eval_summaries = []
    if ckpt_final:
        eval_summaries.append(
            run_eval_job(
                checkpoint_path=ckpt_final,
                algorithm=algorithm,
                split="test",
                alpha=EVAL_ALPHA,
                exp_name=f"{QUICK_EXP_NAME}_eval_final_test",
            )
        )
        eval_summaries.append(
            run_eval_job(
                checkpoint_path=ckpt_final,
                algorithm=algorithm,
                split="all",
                alpha=EVAL_ALPHA,
                exp_name=f"{QUICK_EXP_NAME}_eval_final_all",
            )
        )
    if ckpt_best:
        eval_summaries.append(
            run_eval_job(
                checkpoint_path=ckpt_best,
                algorithm=algorithm,
                split="all",
                alpha=EVAL_ALPHA,
                exp_name=f"{QUICK_EXP_NAME}_eval_best_all",
            )
        )

    QUICK_RUN_ARTIFACTS = {
        "train_summary": quick_summary,
        "eval_summaries": eval_summaries,
    }

    print("quick run complete")
    print(json.dumps(QUICK_RUN_ARTIFACTS, indent=2))
else:
    print("QUICK_MODE is False; skipping quick path.")


## Optional Full Sweep (Local Sequential API)

Set `RUN_FULL_SWEEP=True` to execute the full 84-config grid locally in notebook order.


In [ ]:
FULL_SWEEP_RUNS: list[dict] = []

if RUN_FULL_SWEEP:
    jobs = expand_with_seeds(full_grid_configs, N_SEEDS)
    if FULL_SWEEP_LIMIT is not None:
        jobs = jobs[: int(FULL_SWEEP_LIMIT)]

    print(f"launching {len(jobs)} jobs...")
    for i, job in enumerate(jobs, start=1):
        summary, _ = run_train_job(
            exp_name=job["exp_name"],
            seed=int(job["seed"]),
            base_overrides=full_grid_base_overrides,
            extra_overrides=job["extra_overrides"],
            verbose=False,
        )
        FULL_SWEEP_RUNS.append(summary)

        if i % 5 == 0 or i == len(jobs):
            print(f"completed {i}/{len(jobs)} jobs")

    print("full local sweep complete")
else:
    print("Local full sweep not requested.")


## External Cluster Pickles (No SLURM Required)

You can train on a cluster and copy artifacts locally for notebook analysis:
- Copy checkpoints (`*_final.pkl`, `*_best.pkl`) into `EXTERNAL_CKPT_DIR`.
- Optionally copy JSONL run records into `EXTERNAL_RESULTS_DIR` for table aggregation.

Example copy commands from your local machine:
- `rsync -av user@cluster:/path/to/iro_exp/ckpts/*.pkl notebooks/imported_checkpoints/`
- `rsync -av user@cluster:/path/to/iro_exp/results/<exp_name>/*.jsonl notebooks/imported_results/`


In [ ]:
def discover_checkpoint_pairs(ckpt_dir: Path) -> list[tuple[Path, Path]]:
    pairs: list[tuple[Path, Path]] = []
    for final_path in sorted(ckpt_dir.glob("*_final.pkl")):
        best_path = final_path.with_name(final_path.name.replace("_final.pkl", "_best.pkl"))
        if best_path.exists():
            pairs.append((final_path, best_path))
    return pairs


EXTERNAL_CKPT_PAIRS = discover_checkpoint_pairs(EXTERNAL_CKPT_DIR_PATH) if USE_EXTERNAL_PICKLES else []
print("external checkpoint pairs discovered:", len(EXTERNAL_CKPT_PAIRS))
if EXTERNAL_CKPT_PAIRS:
    first_final, first_best = EXTERNAL_CKPT_PAIRS[0]
    print("first pair:", first_final.name, "<->", first_best.name)
else:
    print("No external checkpoint pairs found yet.")


## JSONL Aggregation and Model Selection

This cell mirrors the repository collector logic (`group by algorithm + args_id`, select by env=0.9).


In [ ]:
def env_key(value: float, *, ms_type: str) -> str:
    return f"{value:g}_acc_{ms_type}"


def is_oracle_row(record: dict) -> bool:
    if str(record.get("algorithm", "")).lower() != "erm":
        return False
    args = record.get("args") or {}
    data = args.get("data") or {}
    envs = data.get("cmnist_train_envs")
    if isinstance(envs, str):
        return envs.strip().lower() == "gray"
    if isinstance(envs, list):
        return [float(x) for x in envs] == [0.5, 0.5]
    return False


def canonical_alg(record: dict) -> str:
    if is_oracle_row(record):
        return "oracle"
    name = str(record.get("algorithm", "")).lower()
    if name == "inftask":
        return "inf-task"
    if name == "iid":
        return "bayes_erm_iid"
    return name


def _load_records_from_jsonl_paths(paths: list[Path]) -> list[dict]:
    records: list[dict] = []
    for path in paths:
        if not path.is_file() or path.stat().st_size == 0:
            continue
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if rec.get("status") != "ok":
                    continue
                if "args_id" not in rec or "algorithm" not in rec:
                    continue
                records.append(rec)
    return records


def iter_result_jsonl_paths(output_root: Path):
    results_root = output_root / "results"
    if not results_root.exists():
        return []
    return sorted(results_root.glob("*/*.jsonl"))


def load_records_from_output_root(output_root: Path) -> list[dict]:
    return _load_records_from_jsonl_paths(list(iter_result_jsonl_paths(output_root)))


def load_records_from_flat_jsonl_dir(results_dir: Path) -> list[dict]:
    if not results_dir.exists():
        return []
    return _load_records_from_jsonl_paths(sorted(results_dir.glob("*.jsonl")))


def group_by_alg_args(records: list[dict]) -> dict[str, dict[str, list[dict]]]:
    grouped: dict[str, dict[str, list[dict]]] = {}
    for rec in records:
        alg = canonical_alg(rec)
        args_id = str(rec["args_id"])
        grouped.setdefault(alg, {}).setdefault(args_id, []).append(rec)
    return grouped


def pick_best_args_id(by_args: dict[str, list[dict]], *, select_env: float, ms_type: str) -> str | None:
    key = env_key(select_env, ms_type=ms_type)
    best_id = None
    best_score = None

    for args_id, recs in by_args.items():
        vals = [float(r[key]) for r in recs if key in r]
        if not vals:
            continue
        score = mean(vals)
        if best_score is None or score > best_score:
            best_score = score
            best_id = args_id
    return best_id


def summarize_env_stats(recs: list[dict], envs: list[float], *, ms_type: str) -> dict:
    env_labels = [f"{env:g}" for env in envs]
    means = []
    stds = []
    strings = []

    for env in envs:
        key = env_key(env, ms_type=ms_type)
        vals = [float(r[key]) for r in recs if key in r]
        if not vals:
            means.append(float("nan"))
            stds.append(float("nan"))
            strings.append("NA")
            continue
        mu = mean(vals)
        sd = stdev(vals) if len(vals) > 1 else 0.0
        means.append(mu)
        stds.append(sd)
        strings.append(f"{mu:.3f} +- {sd:.3f}")

    return {
        "env_labels": env_labels,
        "means": means,
        "stds": stds,
        "strings": strings,
    }


def build_model_selection_summary(
    records: list[dict],
    *,
    envs: list[float],
    model_selection_env: float,
    ms_type: str,
    allowed_algs: set[str] | None = None,
) -> tuple[dict[str, dict], dict[str, list[dict]]]:
    grouped = group_by_alg_args(records)
    rows: dict[str, dict] = {}
    selected_records: dict[str, list[dict]] = {}

    for alg, by_args in grouped.items():
        if allowed_algs and alg not in allowed_algs:
            continue

        best_args_id = pick_best_args_id(by_args, select_env=model_selection_env, ms_type=ms_type)
        if best_args_id is None:
            continue

        recs = by_args[best_args_id]
        rows[alg] = summarize_env_stats(recs, envs, ms_type=ms_type)
        rows[alg]["best_args_id"] = best_args_id
        rows[alg]["num_seeds"] = len(recs)
        selected_records[alg] = recs

    return rows, selected_records


def print_model_selection_table(rows: dict[str, dict], *, envs: list[float]) -> None:
    headers = [f"{e:g}" for e in envs]
    col0w = max([len("algorithm")] + [len(a) for a in rows.keys()]) if rows else len("algorithm")
    colw = 14

    def fmt(parts: list[str]) -> str:
        return "  ".join([parts[0].ljust(col0w)] + [p.ljust(colw) for p in parts[1:]])

    print(fmt(["algorithm"] + headers))
    for alg in sorted(rows.keys()):
        print(fmt([alg] + rows[alg]["strings"]))


## Paper-Style Table (`0.0..1.0`) and Summary View

Switch `MODEL_SELECTION_TYPE` between `best` and `final` to compare selection modes.


In [ ]:
ALL_RECORDS = load_records_from_output_root(OUTPUT_ROOT_PATH)
print("local records loaded:", len(ALL_RECORDS))

if USE_EXTERNAL_PICKLES:
    external_records = load_records_from_flat_jsonl_dir(EXTERNAL_RESULTS_DIR_PATH)
    if external_records:
        print("external records loaded:", len(external_records))
        ALL_RECORDS.extend(external_records)

print("total records used:", len(ALL_RECORDS))

MODEL_ROWS, SELECTED_RECORDS = build_model_selection_summary(
    ALL_RECORDS,
    envs=TEST_ENVS_ALL,
    model_selection_env=MODEL_SELECTION_ENV,
    ms_type=MODEL_SELECTION_TYPE,
)

if MODEL_ROWS:
    print_model_selection_table(MODEL_ROWS, envs=TEST_ENVS_ALL)
else:
    print("No model-selection rows available yet. Run quick/full sweep and/or copy JSONL files from cluster.")

# Keep these globals for plotting cells
CURVE_ROWS = MODEL_ROWS


## Checkpoint Diagnostics (`final` vs `best`)

Loads `.pkl` checkpoints and reports state-dict norms and delta statistics.


In [ ]:
def load_state_dict_from_checkpoint(path: Path) -> dict[str, torch.Tensor]:
    payload = torch.load(path, map_location="cpu")
    if isinstance(payload, dict) and "state_dict" in payload and isinstance(payload["state_dict"], dict):
        return payload["state_dict"]
    if isinstance(payload, dict):
        return payload
    raise ValueError(f"Unsupported checkpoint payload type: {type(payload)}")


def checkpoint_l2_norm(state_dict: dict[str, torch.Tensor]) -> float:
    sq_sum = 0.0
    for tensor in state_dict.values():
        t = tensor.detach().float().cpu()
        sq_sum += float(torch.sum(t * t).item())
    return math.sqrt(sq_sum)


def checkpoint_delta_stats(final_sd: dict[str, torch.Tensor], best_sd: dict[str, torch.Tensor]) -> dict:
    keys = sorted(set(final_sd.keys()) & set(best_sd.keys()))
    per_param_l2 = {}
    total_sq = 0.0

    for key in keys:
        diff = final_sd[key].detach().float().cpu() - best_sd[key].detach().float().cpu()
        l2 = float(torch.linalg.norm(diff).item())
        per_param_l2[key] = l2
        total_sq += float(torch.sum(diff * diff).item())

    total_l2 = math.sqrt(total_sq)
    top = sorted(per_param_l2.items(), key=lambda kv: kv[1], reverse=True)[:10]

    return {
        "shared_params": len(keys),
        "delta_l2_total": total_l2,
        "top_param_deltas": top,
    }


def latest_checkpoint_pair(output_root: Path) -> tuple[Path | None, Path | None]:
    ckpt_dir = output_root / "ckpts"
    if not ckpt_dir.exists():
        return None, None

    finals = sorted(ckpt_dir.glob("*_final.pkl"))
    for final_path in reversed(finals):
        best_path = final_path.with_name(final_path.name.replace("_final.pkl", "_best.pkl"))
        if best_path.exists():
            return final_path, best_path
    return None, None


def summarize_checkpoint_pair(label: str, final_path: Path, best_path: Path) -> dict:
    final_sd = load_state_dict_from_checkpoint(final_path)
    best_sd = load_state_dict_from_checkpoint(best_path)

    norm_final = checkpoint_l2_norm(final_sd)
    norm_best = checkpoint_l2_norm(best_sd)
    delta = checkpoint_delta_stats(final_sd, best_sd)

    return {
        "label": label,
        "final_path": str(final_path),
        "best_path": str(best_path),
        "norm_final": norm_final,
        "norm_best": norm_best,
        "delta_l2_total": delta["delta_l2_total"],
        "top_param_deltas": delta["top_param_deltas"],
    }


checkpoint_pairs: list[tuple[str, Path, Path]] = []

# Quick-run pair if available
if QUICK_RUN_ARTIFACTS:
    train_summary = QUICK_RUN_ARTIFACTS.get("train_summary", {})
    final_candidate = train_summary.get("ckpt_final")
    best_candidate = train_summary.get("ckpt_best")
    if final_candidate and best_candidate:
        checkpoint_pairs.append(("quick_run", Path(final_candidate), Path(best_candidate)))

# Latest local pair fallback
if not checkpoint_pairs:
    final_path, best_path = latest_checkpoint_pair(OUTPUT_ROOT_PATH)
    if final_path is not None and best_path is not None:
        checkpoint_pairs.append(("latest_local", final_path, best_path))

# External copied pairs
if USE_EXTERNAL_PICKLES and EXTERNAL_CKPT_PAIRS:
    for idx, (final_path, best_path) in enumerate(EXTERNAL_CKPT_PAIRS, start=1):
        checkpoint_pairs.append((f"external_{idx:03d}", final_path, best_path))

CHECKPOINT_DIAGNOSTICS = []
if not checkpoint_pairs:
    print("No checkpoint pairs available. Run quick path or copy external pickles first.")
else:
    for label, final_path, best_path in checkpoint_pairs:
        summary = summarize_checkpoint_pair(label, final_path, best_path)
        CHECKPOINT_DIAGNOSTICS.append(summary)

    print(f"diagnosed {len(CHECKPOINT_DIAGNOSTICS)} checkpoint pairs")
    for summary in CHECKPOINT_DIAGNOSTICS[:5]:
        print("-", summary["label"])
        print("  final:", summary["final_path"])
        print("  best :", summary["best_path"])
        print(f"  L2(final)={summary['norm_final']:.6f}  L2(best)={summary['norm_best']:.6f}  delta={summary['delta_l2_total']:.6f}")
        print("  top deltas:")
        for key, value in summary["top_param_deltas"][:3]:
            print(f"    {key}: {value:.6f}")


## Extended Visuals

- Environment-shift curves
- Seed-variance bands
- Average `best - final` deltas per algorithm


In [ ]:
def compute_best_minus_final_delta(selected_records: dict[str, list[dict]], envs: list[float]) -> dict[str, float]:
    out: dict[str, float] = {}
    for alg, recs in selected_records.items():
        per_env_delta = []
        for env in envs:
            best_key = f"{env:g}_acc_best"
            final_key = f"{env:g}_acc_final"
            best_vals = [float(r[best_key]) for r in recs if best_key in r]
            final_vals = [float(r[final_key]) for r in recs if final_key in r]
            if best_vals and final_vals:
                per_env_delta.append(mean(best_vals) - mean(final_vals))
        if per_env_delta:
            out[alg] = mean(per_env_delta)
    return out


if not HAS_MPL:
    print("matplotlib is not installed; skipping plots.")
elif not CURVE_ROWS:
    print("No curve rows available yet; run quick path and/or full sweep first.")
else:
    envs = np.array(TEST_ENVS_ALL, dtype=float)

    # Plot 1: env curves with mean accuracy
    plt.figure(figsize=(10, 5))
    for alg in sorted(CURVE_ROWS.keys()):
        means = np.array(CURVE_ROWS[alg]["means"], dtype=float)
        plt.plot(envs, means, marker="o", linewidth=1.5, label=alg)
    plt.title(f"CMNIST env-shift curves ({MODEL_SELECTION_TYPE} selection)")
    plt.xlabel("Environment color-shift value")
    plt.ylabel("Accuracy")
    plt.ylim(0.0, 1.0)
    plt.grid(alpha=0.2)
    plt.legend(loc="best", fontsize=8)
    plt.show()

    # Plot 2: variance bands for up to top-6 algorithms (by mean accuracy)
    avg_scores = []
    for alg, stats in CURVE_ROWS.items():
        vals = [v for v in stats["means"] if not math.isnan(v)]
        avg_scores.append((alg, float(np.mean(vals)) if vals else float("-inf")))
    top_algs = [alg for alg, _ in sorted(avg_scores, key=lambda kv: kv[1], reverse=True)[:6]]

    plt.figure(figsize=(10, 5))
    for alg in top_algs:
        means = np.array(CURVE_ROWS[alg]["means"], dtype=float)
        stds = np.array(CURVE_ROWS[alg]["stds"], dtype=float)
        lower = np.clip(means - stds, 0.0, 1.0)
        upper = np.clip(means + stds, 0.0, 1.0)
        plt.plot(envs, means, linewidth=1.8, label=alg)
        plt.fill_between(envs, lower, upper, alpha=0.15)
    plt.title("Seed variance bands (top algorithms)")
    plt.xlabel("Environment color-shift value")
    plt.ylabel("Accuracy")
    plt.ylim(0.0, 1.0)
    plt.grid(alpha=0.2)
    plt.legend(loc="best", fontsize=8)
    plt.show()

    # Plot 3: avg best-final delta per algorithm
    deltas = compute_best_minus_final_delta(SELECTED_RECORDS, TEST_ENVS_ALL)
    if deltas:
        ordered = sorted(deltas.items(), key=lambda kv: kv[1], reverse=True)
        labels = [k for k, _ in ordered]
        values = [v for _, v in ordered]

        plt.figure(figsize=(10, 5))
        plt.bar(labels, values)
        plt.axhline(0.0, color="black", linewidth=1)
        plt.title("Average (best - final) accuracy delta across envs")
        plt.xlabel("Algorithm")
        plt.ylabel("Mean delta")
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y", alpha=0.2)
        plt.show()
    else:
        print("Not enough best/final metrics available to compute delta bar chart.")


## Interpretation and Next Steps

Fill this after runs complete:
- Quick-path run status (train + eval):
- Which algorithms are strongest at env=0.9 model selection:
- How sensitive each algorithm is across env shifts 0.0..1.0:
- Whether `best` vs `final` checkpoint choice materially changes conclusions:

External-pickle analysis checklist:
- Confirm copied checkpoint pairs are discovered in `EXTERNAL_CKPT_DIR`.
- (Optional) copy JSONL files into `EXTERNAL_RESULTS_DIR` to include cluster runs in tables/plots.
- Re-run the table and checkpoint diagnostics cells to merge local + external evidence.

Placeholder for iWildCam parity notebook:
- Reuse this structure for data prep, grouped minibatches, WILDS split metrics, and checkpoint diagnostics.
